# Hamiltonians

In general, we can write the Hamiltonian as a linear combination of $M$ local operators:

$H=\sum_m^M{c_m O_m}$

where the coefficients, $c_m$ are real.

While there are other packages that provide useful tools for constructing and manipulating Hamiltonians, Quax keeps this very basic. Hamiltonians are nothing more than `Operators`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import quax as qx
import jax.numpy as jnp
from quax.gates import I, X, Y, Z

We can construct a basic Hamiltonian using the fundamental operators. For example the Hamiltonian `XX+YY` can be constructed with

In [3]:
hamiltonian = (X | X) + (Y | Y)
print(hamiltonian)

Unitary(dims=((2, 2), (2, 2)), shape=(2, 2, 2, 2))


The resulting object here is a `Unitary`, since `XX` and `YY` are both unitaries. But generally, we want to include some coefficients so we'll end up with an `Operator`.

_Note: In python the `|` (tensor) operator is appplied _after_ addition and subtraction. So be careful to use parantheses._

In [4]:
theta = jnp.pi / 4
hamiltonian = theta * ((X | X) + (Y | Y))
print(hamiltonian)

Kraus(dims=((2, 2), (2, 2)), shape=(2, 2, 2, 2))


In Quax, operators are not attached to qubit indices. So in order to construct a large Hamiltonian it's necessary to write the whole Pauli string. For example, we extend our operators to 5 qubits here.

In [5]:
hamiltonian = theta * ((I | X | X | I | I) + (I | Y | Y | I | I))
print(hamiltonian)

Kraus(dims=((2, 2, 2, 2, 2), (2, 2, 2, 2, 2)), shape=(2, 2, 2, 2, 2, 2, 2, 2, 2, 2))


## Exponentiation

Our favourite thing to do with Hamiltonians is to exponentiate them. Exponentiated `Operator` objects produce `Unitary` objects, just as in life.

$ U = e^{iH} $

Quax offers two functions to do this: `exp` and `cis`. If you aren't familiar with `cis`, it's shorthand for $e^{ix}$, `cis(x)` is equivalent to `exp(1j*x)`.

It's often preferable to use `cis` since we can be sure that the output is a unitary, while the output of `exp` could be a more general object.

### Constructing gates

Hamiltonian exponentation is very general, but one of the basic tasks we can do with it is construct gates.

For example, the single-qubit rotations `RX`, `RY` and `RZ` are equivalent to,

$\text{RX}(\theta) = e^{-i \frac{\theta}{2} \text{X}}$

$\text{RY}(\theta) = e^{-i \frac{\theta}{2} \text{Y}}$

$\text{RZ}(\theta) = e^{-i \frac{\theta}{2} \text{Z}}$

which we can translate to code like so

In [6]:
theta = jnp.pi / 2
SX = qx.cis((-theta / 2) * X)
SY = qx.cis((-theta / 2) * Y)
SZ = qx.cis((-theta / 2) * Z)

for name, gate in [("RX(𝜋/2)", SX), ("RY(𝜋/2)", SY), ("RZ(𝜋/2)", SZ)]:
    print(f"{name} =")
    print(jnp.round(gate.matrix, 6))
    print()

RX(𝜋/2) =
[[0.707107+0.j       0.      -0.707107j]
 [0.      -0.707107j 0.707107+0.j      ]]

RY(𝜋/2) =
[[ 0.707107+0.j -0.707107+0.j]
 [ 0.707107+0.j  0.707107+0.j]]

RZ(𝜋/2) =
[[1.+0.j 0.+0.j]
 [0.+0.j 0.+1.j]]



### 2Q gates

We can also construct 2Q gates using exponentation. Some common examples are:


$\mathrm{CZ} = e^{-i \frac{\pi}{4}(ZZ - ZI - IZ)}$

$\mathrm{CNOT} = e^{-i \frac{\pi}{4}(ZX - ZI - IX)}$

$\mathrm{ISWAP} = e^{+i \frac{\pi}{4}(XX + YY)}$

$\mathrm{FSIM} = e^{-i \frac{\theta}{2}(ZZ - ZI - IZ) + \frac{\phi}{2}(ZZ - ZI - IZ)}$

Note that we use the Rigetti conventions here which may differ by a sign or constant factor from other definitions.

In [7]:
hamiltonian = (+jnp.pi / 4) * ((Z | Z) - (I | Z) - (Z | I))
cz = qx.cis(hamiltonian)
print("CZ = ")
print(f"{jnp.round(cz.matrix, 6)}")
print("")

hamiltonian = (+jnp.pi / 4) * ((Z | X) - (I | X) - (Z | I))
cx = qx.cis(hamiltonian)
print("CX = ")
print(f"{jnp.round(cx.matrix, 6)}")
print("")

hamiltonian = (+jnp.pi / 4) * ((X | X) + (Y | Y))
iswap = qx.cis(hamiltonian)
print("ISWAP = ")
print(f"{jnp.round(iswap.matrix, 6)}")
print("")

hamiltonian = (+jnp.pi / 4) * ((X | X) + (Y | Y)) + (+jnp.pi / 4 / 6) * ((Z | Z) - (I | Z) - (Z | I))
fsim = qx.cis(hamiltonian)
print("FSIM(𝜃=𝜋, 𝜙=𝜋/6) = ")
print(f"{jnp.round(fsim.matrix, 6)}")
print("")

CZ = 
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j -1.-0.j]]

CX = 
[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.-0.j  1.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j -0.-0.j]]

ISWAP = 
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+1.j 0.+0.j]
 [0.+0.j 0.+1.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]

FSIM(𝜃=𝜋, 𝜙=𝜋/6) = 
[[ 1.      -0.j   0.      +0.j   0.      +0.j   0.      +0.j ]
 [ 0.      +0.j  -0.      -0.j  -0.      +1.j   0.      +0.j ]
 [ 0.      +0.j  -0.      +1.j  -0.      +0.j   0.      +0.j ]
 [ 0.      +0.j   0.      +0.j   0.      +0.j   0.866025+0.5j]]



## Ensembles of Hamiltonians

Like all objects in Quax, exponentation can be done on ensembles.

In [8]:
theta = jnp.linspace(0, jnp.pi / 4, 24)
unitaries = qx.cis(theta * ((X | X) + (Y | Y)))
print(unitaries)

Unitary(dims=((2, 2), (2, 2)), ensemble_size=(24,), shape=(24, 2, 2, 2, 2))
